# LVS Falsifiability Test: Global Stationarity vs $\alpha_s$

## Context
This notebook formally tests a direct mathematical formulation of the LVS (Latent Vacuum Stationarity) theory. The core premise of LVS is that observable reality corresponds to stationary fixed points in the quantum vacuum. 

If we interpret this to mean that the *overall* Renormalization Group (RG) flow of the Standard Model must minimize its variation (i.e. maximize stationarity) between the electroweak scale ($M_Z$) and the Planck scale ($M_{Pl}$), we can test this hypothesis. We leave all parameters fixed at their measured values except for the strong coupling constant $\alpha_s(M_Z)$, and we search for the value of $\alpha_s(M_Z)$ that minimizes the "length" of the RG flow trajectory.

## The Calculations
We evaluate two specific metrics representing the "lack of stationarity" (or movement) of the flow:
1. **Relative Length (Zamolodchikov-like Metric):** The integral of the relative variation of the couplings $\sigma(\mu) = \sqrt{\sum (\frac{\beta_i}{g_i})^2}$ over the flow.
2. **Euclidean Length:** The standard Euclidean integral of the $\beta$-functions $\sqrt{\sum \beta_i^2}$ over the flow.

If the "naive" LVS hypothesis is correct, the mathematically optimal $\alpha_s(M_Z)$ that minimizes these lengths should closely match the experimentally measured value of $\alpha_s(M_Z) = 0.1179 \pm 0.0010$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Physical constants (PDG 2024)
M_Z = 91.1876
M_Pl = 1.22e19
M_t = 172.76
v_EW = 246.22

alpha_em_MZ_inv = 127.951
sin2_theta_W = 0.23122
alpha_em_MZ = 1.0 / alpha_em_MZ_inv
alpha_1_MZ = (5.0/3.0) * alpha_em_MZ / (1.0 - sin2_theta_W)
alpha_2_MZ = alpha_em_MZ / sin2_theta_W

g1_MZ = np.sqrt(4.0 * np.pi * alpha_1_MZ)
g2_MZ = np.sqrt(4.0 * np.pi * alpha_2_MZ)
yt_MZ = np.sqrt(2.0) * M_t / v_EW

B_SM = np.array([41.0/10.0, -19.0/6.0, -7.0])

def beta_SM_1loop(t, y):
    pref = 1.0 / (16.0 * np.pi**2)
    g1, g2, g3, yt = y
    dg1 = B_SM[0] * g1**3 * pref
    dg2 = B_SM[1] * g2**3 * pref
    dg3 = B_SM[2] * g3**3 * pref
    dyt = yt * pref * ((9.0/2.0)*yt**2 - (17.0/20.0)*g1**2 - (9.0/4.0)*g2**2 - 8.0*g3**2)
    return [dg1, dg2, dg3, dyt]

def compute_lengths(alpha_s_mz):
    g3_MZ = np.sqrt(4.0 * np.pi * alpha_s_mz)
    y0 = [g1_MZ, g2_MZ, g3_MZ, yt_MZ]
    t_span = (np.log(M_Z), np.log(M_Pl))
    
    sol = solve_ivp(beta_SM_1loop, t_span, y0, dense_output=True, rtol=1e-8, atol=1e-10)
    t = np.linspace(t_span[0], t_span[1], 1000)
    y = sol.sol(t)
    
    g1, g2, g3, yt = y
    pref = 1.0 / (16.0 * np.pi**2)
    dg1 = B_SM[0] * g1**3 * pref
    dg2 = B_SM[1] * g2**3 * pref
    dg3 = B_SM[2] * g3**3 * pref
    dyt = yt * pref * ((9.0/2.0)*yt**2 - (17.0/20.0)*g1**2 - (9.0/4.0)*g2**2 - 8.0*g3**2)
    
    ds_euclid = np.sqrt(dg1**2 + dg2**2 + dg3**2 + dyt**2)
    length_euclidean = np.trapezoid(ds_euclid, t)
    
    ds_rel = np.sqrt((dg1/g1)**2 + (dg2/g2)**2 + (dg3/g3)**2 + (dyt/yt)**2)
    length_rel = np.trapezoid(ds_rel, t)
    
    return length_euclidean, length_rel

alphas_test = np.linspace(0.04, 0.16, 100)
lengths_e = []
lengths_r = []

for a_s in alphas_test:
    le, lr = compute_lengths(a_s)
    lengths_e.append(le)
    lengths_r.append(lr)

best_e = alphas_test[np.argmin(lengths_e)]
best_r = alphas_test[np.argmin(lengths_r)]

plt.figure(figsize=(10, 5))
plt.plot(alphas_test, lengths_r, label='Relative Length (Zamolodchikov-like)')
plt.axvline(best_r, color='r', linestyle='--', label=f'Min Rel Length = {best_r:.4f}')
plt.axvline(0.1179, color='g', linestyle='-', label='Measured $\alpha_s$ = 0.1179')
plt.xlabel('$\alpha_s(M_Z)$')
plt.ylabel('RG Flow Length')
plt.title('Stationarity of RG Flow vs $\alpha_s$')
plt.legend()
plt.grid(True)
plt.show()

print(f"Optimal alpha_s for Euclidean length: {best_e:.4f}")
print(f"Optimal alpha_s for Relative length: {best_r:.4f}")
print(f"Measured alpha_s: 0.1179")


## Conclusion of the Test
The results are unambiguous:
- Optimal $\alpha_s(M_Z)$ for global stationarity (Relative Length): **$\approx 0.056$**
- Optimal $\alpha_s(M_Z)$ for Euclidean Length: **$\approx 0.058$**
- Measured $\alpha_s(M_Z)$ in reality: **$0.1179$**

**Scientific Conclusion:**
The naive formulation of LVS, which posits that the Standard Model parameters are selected to globally minimize the RG running over the entire trajectory, is **falsified**. 
The actual universe is not the path of "least resistance" or "maximal global stationarity" in the space of couplings.

**Consequences for LVS:**
This negative result is extremely valuable as it formally closes a theoretical dead-end. The true LVS condition, if it exists, must be:
1. A strict, localized condition specifically at the Planck scale (e.g., $\lambda(M_{Pl})=0$ exactly coinciding with $\beta_\lambda(M_{Pl})=0$).
2. Or a constraint arising only within the full quantum gravity framework (Asymptotic Safety) when the running of the gravitational coupling and cosmological constant are explicitly included.
